# Unit 3: Transformer Architectures for Audio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

**Course**: [HuggingFace Audio Transformers Course - Unit 3](https://huggingface.co/learn/audio-course/chapter3)

**Topics covered:**
- Encoder-only, decoder-only, and encoder-decoder architectures
- Waveform input models (Wav2Vec2, HuBERT)
- Spectrogram input models (Whisper)
- CTC vs seq2seq approaches for ASR
- Pre-training strategies for audio models

## Setup

In [ ]:
!pip install -q transformers datasets librosa torch

## 1. Audio Transformer Architecture Overview

All audio transformers share the same core architecture. The differences are in:
- **Input processing**: waveform vs spectrogram
- **Architecture type**: encoder-only, decoder-only, encoder-decoder
- **Output head**: classification, CTC, seq2seq, spectrogram generation

### Architecture Types

| Type | Models | Best For |
|------|--------|----------|
| Encoder-only | Wav2Vec2, HuBERT, Audio Spectrogram Transformer | Classification, ASR (with CTC) |
| Encoder-decoder | Whisper, SpeechT5 | ASR (seq2seq), TTS |
| Decoder-only | (less common in audio) | Audio generation |

## 2. Waveform Input Models

Models like **Wav2Vec2** and **HuBERT** take raw waveform as input. A CNN feature encoder converts the waveform into embeddings (one per ~25ms of audio).

In [ ]:
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor
import torch
import librosa

# Load model and feature extractor
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-base")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")

# Load audio
audio, sr = librosa.load(librosa.ex('trumpet'), sr=16000)
# Take first 3 seconds
audio = audio[:16000 * 3]

# Process
inputs = feature_extractor(audio, sampling_rate=16000, return_tensors="pt")
print(f"Input shape: {inputs.input_values.shape}")

with torch.no_grad():
    outputs = model(**inputs)

print(f"Hidden states shape: {outputs.last_hidden_state.shape}")
print(f"Sequence length reduced from {len(audio)} samples to {outputs.last_hidden_state.shape[1]} embeddings")

## 3. Spectrogram Input Models

Models like **Whisper** first convert audio to a log-mel spectrogram, then process it. This gives a much shorter sequence.

In [ ]:
from transformers import WhisperModel, WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-tiny")
model = WhisperModel.from_pretrained("openai/whisper-tiny")

inputs = feature_extractor(audio, sampling_rate=16000, return_tensors="pt")
print(f"Mel spectrogram shape: {inputs.input_features.shape}")
print("Shape is (batch, mel_bins=80, time_steps=3000) - always 30s padded")

# Get encoder outputs
with torch.no_grad():
    encoder_outputs = model.encoder(inputs.input_features)

print(f"Encoder hidden states: {encoder_outputs.last_hidden_state.shape}")

## 4. CTC vs Seq2Seq for ASR

Two main approaches for speech recognition:

**CTC (Connectionist Temporal Classification)**:
- Encoder-only, predicts one token per time step
- Uses a special blank token for alignment
- Fast inference, but struggles with spelling/punctuation
- Example: Wav2Vec2 with CTC head

**Seq2Seq (Sequence-to-Sequence)**:
- Encoder-decoder, generates tokens autoregressively
- Better at punctuation, casing, and formatting
- Slower inference due to autoregressive decoding
- Example: Whisper

In [ ]:
from transformers import pipeline
from datasets import load_dataset

# Load a speech sample
minds14 = load_dataset("PolyAI/minds14", name="en-US", split="train", trust_remote_code=True)
sample = minds14[0]

# CTC-based ASR (Wav2Vec2)
asr_ctc = pipeline("automatic-speech-recognition", model="facebook/wav2vec2-base-960h")
result_ctc = asr_ctc(sample["audio"]["array"])
print(f"CTC (Wav2Vec2):  {result_ctc['text']}")

# Seq2Seq ASR (Whisper)
asr_seq2seq = pipeline("automatic-speech-recognition", model="openai/whisper-tiny")
result_seq2seq = asr_seq2seq(sample["audio"]["array"])
print(f"Seq2Seq (Whisper): {result_seq2seq['text']}")

print(f"\nGround truth: {sample.get('transcription', 'N/A')}")

## Key Takeaways

- Audio transformers share the same core architecture, differing mainly in input/output processing
- Waveform models (Wav2Vec2, HuBERT) use a CNN to convert raw audio to embeddings
- Spectrogram models (Whisper) convert audio to log-mel spectrograms first, giving shorter sequences
- CTC is fast but less capable; seq2seq is slower but handles formatting better
- Pre-training on unlabeled audio (self-supervised) enables strong performance with limited labeled data

**Next**: [Unit 4 - Music Genre Classifier](../unit_4/04_music_genre_classifier.ipynb) (Hands-on Exercise!)